# Cold inference demonstration: W6 final
Use a Conda kernel and set `SOLAR_IMAGES_DIR` and `SOLAR_OUTPUT_DIR`. This notebook verifies every weight shard, reconstructs the models, then executes the eight frozen W6 stages through `scripts/demo_w6.py`. The final output is `w6.csv`. Set `SOLAR_DENSE_PYTHON`, `SOLAR_SUPPORT_PYTHON`, `SOLAR_SCORE_PYTHON`, and `SOLAR_REFINER_PYTHON` for separate Conda environments. The launcher cleans inherited kernel settings and repairs an empty inherited CUDA device mask for GPU stages. Run every cell; stage logs and completion receipts are saved under the output cache. No Kaggle submission is performed.


In [ ]:
from pathlib import Path
import os, sys, subprocess, csv, time, platform
root = Path.cwd().resolve()
if root.name == "notebooks": root = root.parent
sys.path.insert(0, str(root))
from merge_weights import merge
images = Path(os.environ["SOLAR_IMAGES_DIR"]).resolve()
output = Path(os.environ.get("SOLAR_OUTPUT_DIR", "run_demo")).resolve()
output.mkdir(parents=True, exist_ok=True)
models = output / "models"
started = time.monotonic()
merge(root, models, os.environ.get("SOLAR_RELEASE_RAW_BASE_URL"))
assert images.is_dir()
print("SHARDS_AND_MODELS_SHA_VERIFIED", platform.machine())


In [ ]:
command = [sys.executable, str(root / "scripts/demo_w6.py"),
           "--images", str(images), "--models", str(models),
           "--cache", str(output / "cache"), "--output", str(output / "w6.csv"),
           "--final", "w6", "--dense-long2x-tta", "--postcal-w2", "--device", "cuda"]
subprocess.run(command, cwd=root, check=True)


In [ ]:
for filename in ("w6.csv",):
    with (output / filename).open(newline="") as stream:
        reader = csv.DictReader(stream)
        assert reader.fieldnames == ["filament_id", "segmentation_rle"]
        rows = list(reader)
    assert len({row["filament_id"] for row in rows}) == len(rows)
    print(filename, len(rows))
print("TOTAL_SECONDS", time.monotonic() - started)
